<a href="https://colab.research.google.com/github/zhangling297/Data-Science-For-Beginners/blob/main/Mat599_Pro5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Recommendations for Developing Contrastive Learning

1. **Data Augmentation**: This is the heart of most contrastive methods (like SimCLR). You need to generate two different augmented views of the same image. Strong augmentations (color jitter, random crop, Gaussian blur) are critical.
2. **Encoder**: A standard backbone like ResNet-18 or ResNet-50 to extract features.
3. **Projection Head**: An MLP that maps the encoder representations to the space where the contrastive loss is applied. This prevents the model from collapsing.
4. **Loss Function**: InfoNCE (or NT-Xent) loss is the standard. It treats the two augmented views of the same image as a positive pair and all other images in the batch as negatives.

Below is a basic skeleton for a PyTorch-based contrastive learning setup.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

# 1. Define the Model with a Projection Head
class ContrastiveModel(nn.Module):
    def __init__(self, base_model='resnet18', out_dim=128):
        super(ContrastiveModel, self).__init__()
        # Use a pretrained or untrained ResNet
        self.encoder = models.__dict__[base_model](weights=None)

        # Remove the classification head
        dim_mlp = self.encoder.fc.in_features
        self.encoder.fc = nn.Identity()

        # Add a projection head (MLP)
        self.projector = nn.Sequential(
            nn.Linear(dim_mlp, dim_mlp),
            nn.ReLU(),
            nn.Linear(dim_mlp, out_dim)
        )

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return F.normalize(z, dim=1) # Normalize the outputs

model = ContrastiveModel()
print("Model initialized!")

In [ ]:
# 2. Define the Contrastive Loss (NT-Xent / InfoNCE)
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super(NTXentLoss, self).__init__()
        self.temperature = temperature
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, z_i, z_j):
        batch_size = z_i.shape[0]

        # Concatenate the representations
        z = torch.cat((z_i, z_j), dim=0)

        # Compute similarity matrix
        sim_matrix = torch.matmul(z, z.T) / self.temperature

        # Create labels (each sample's positive pair is batch_size away)
        labels = torch.cat([torch.arange(batch_size) + batch_size, torch.arange(batch_size)], dim=0)
        labels = labels.to(z.device)

        # Mask out self-similarity
        mask = torch.eye(2 * batch_size, dtype=torch.bool).to(z.device)
        sim_matrix.masked_fill_(mask, -9e15)

        loss = self.criterion(sim_matrix, labels)
        return loss

loss_fn = NTXentLoss()
print("Loss function initialized!")

### Next Steps:
To adapt this to your specific dataset:
1. Update your `Dataset` class to return *two* augmented versions of each input.
2. Write a training loop that passes both augmented batches (`x_i`, `x_j`) through the `ContrastiveModel` to get `z_i` and `z_j`.
3. Compute the `NTXentLoss(z_i, z_j)` and backpropagate.

## Project Recommendations: INbreast Tumor Segmentation with U-Net, ViT, and Contrastive Learning

Your project idea addresses a highly relevant clinical problem: **fuzzy boundaries in breast cancer lesions** (benign vs. malignant) using the **INbreast dataset**.

### 1. Historical Context (Who, When, Where)
*   **INbreast Dataset**: Introduced by Inès C. Moreira et al. in **2012** (published in *Academic Radiology*). It's a widely used full-field digital mammography dataset with precise annotations (contours) for masses and calcifications.
*   **U-Net**: Introduced by **Olaf Ronneberger, Philipp Fischer, and Thomas Brox** in **2015** at the **MICCAI** (Medical Image Computing and Computer Assisted Intervention) conference. It became the gold standard for medical image segmentation due to its symmetric encoder-decoder structure and skip connections.
*   **Vision Transformer (ViT)**: Introduced by **Alexey Dosovitskiy et al. (Google Brain)** in late **2020** (published at **ICLR 2021**). It proved that pure transformers applied directly to sequences of image patches can perform exceedingly well on image classification.
*   ***Recommendation - TransUNet***: Instead of just basic ViT, look into **TransUNet** (Chen et al., 2021). It combines the U-Net architecture with Transformers, using ViT as the encoder to capture global context (great for fuzzy boundaries) and U-Net's decoder for precise localization.

### 2. Why this is a good idea
Fuzzy boundaries happen because the contrast between dense breast tissue and tumors is low. Standard CNNs (like basic U-Net) struggle because their receptive field is limited; they only look at local pixel neighborhoods.
**Transformers (ViT)** look at the global context (how the tumor relates to the whole breast tissue structure) using self-attention, making them much better at inferring where a fuzzy boundary *should* be based on global patterns.

### 3. How to implement Contrastive Learning for this task
Standard contrastive learning (like the SimCLR skeleton we built above) works at the **image level**. However, you are doing **segmentation** (pixel-level classification). Here is how you combine them:

**Approach A: Encoder Pre-training (Standard)**
1. Extract ROIs (Regions of Interest) from the INbreast dataset.
2. Apply strong augmentations (blurring, rotations, elastic deformations) to create positive pairs.
3. Pre-train your U-Net or ViT **encoder** using the `ContrastiveModel` and `NTXentLoss` we defined earlier. The model learns to understand breast tissue textures without needing the segmentation masks.
4. Fine-tune the whole model (Encoder + Decoder) using standard segmentation losses (like Dice Loss or Cross Entropy) on the ground truth boundaries.

**Approach B: Dense / Pixel-Level Contrastive Learning (Advanced & Highly Recommended)**
Instead of pulling two whole *images* together, you pull *pixels* of the same class together in the feature space.
1. Pass two augmented views of an image through the network.
2. Instead of a single vector `z`, output a dense feature map.
3. Apply **Pixel-to-Pixel Contrastive Loss**: Take a feature vector for a "Tumor Boundary" pixel in View 1, and make it attract the corresponding "Tumor Boundary" pixel in View 2, while pushing it away from "Background" pixels or "Benign" pixels.
4. **Why this helps your specific problem:** By forcing the model to explicitly push "tumor boundary" features far away from "healthy tissue" features in the latent space, the model naturally learns a sharper boundary, combating the fuzziness of the mammogram.

### Suggested Pipeline:
1. Preprocess INbreast (extract ROIs, normalize).
2. Set up a Hybrid Model (e.g., TransUNet).
3. Apply Pixel-level Contrastive Learning as an auxiliary loss alongside your main Dice Loss during training.

### Alternative: Paste your Kaggle Key directly
If the file uploader is giving you trouble, just paste your username and key in the strings below and run this cell.

In [ ]:
import os

# 1. PASTE YOUR KAGGLE USERNAME AND KEY HERE:
kaggle_username = "lzhanglynn"
kaggle_key = "KGAT_bf99f3af333dc5dd25fbce18b26d2b45"

# 2. Create the .kaggle directory
os.makedirs('/root/.kaggle', exist_ok=True)

# 3. Write the credentials to the kaggle.json file
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')

# 4. Set permissions so Kaggle accepts it
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Kaggle API key successfully configured!")

# 5. Download the notebook and its output files
# The provided link is a notebook, so we use 'kernels output' instead of 'datasets download'
print("Downloading notebook and outputs from Kaggle...")
!kaggle kernels output lethanhnghia/breastcancercnn -p /content/breastcancercnn

print("Notebook and any output files downloaded to /content/breastcancercnn!")

In [ ]:
!kaggle datasets download -d tommyngx/inbreast2012 -p /content/inbreast2012

In [ ]:
!kaggle datasets download -d martholi/inbreast -p /content/inbreast

In [ ]:
from google.colab import drive

# This will prompt you to log in and grant access to your Google Drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil

# Define the destination folder in your Google Drive
drive_folder = '/content/drive/MyDrive/INbreast_Project'
os.makedirs(drive_folder, exist_ok=True)
print(f"Created directory: {drive_folder}")

# Paths to the downloaded zip files
zip1 = '/content/inbreast2012/inbreast2012.zip'
zip2 = '/content/inbreast/inbreast.zip'

# Copy the files to Google Drive
if os.path.exists(zip1):
    print(f"Copying {zip1} to Drive...")
    shutil.copy(zip1, drive_folder)
    print("Done!")

if os.path.exists(zip2):
    print(f"Copying {zip2} to Drive...")
    shutil.copy(zip2, drive_folder)
    print("Done!")

print(f"\nSuccessfully saved the datasets to {drive_folder}!")

In [ ]:
!pip install pydicom -q

In [ ]:
import pydicom
import matplotlib.pyplot as plt
import os
import glob

# Path to the directory where the DICOM files are stored
dcm_dir = '/content/breastcancercnn/ALL-IMGS/'

# Find all DICOM files in the directory
dcm_files = glob.glob(os.path.join(dcm_dir, '*.dcm'))

if dcm_files:
    # Pick the first file as our sample
    sample_file = dcm_files[0]
    print(f"Loading sample image: {os.path.basename(sample_file)}")

    # Read the DICOM file using pydicom
    dicom_data = pydicom.dcmread(sample_file)

    # Extract the pixel array (the actual image)
    image_pixels = dicom_data.pixel_array

    # Display the image using matplotlib
    plt.figure(figsize=(8, 10))
    plt.imshow(image_pixels, cmap='gray') # Display in grayscale
    plt.title(f"INbreast Sample\nShape: {image_pixels.shape}")
    plt.axis('off') # Hide axes for a cleaner look
    plt.show()
else:
    print("Could not find any DICOM files. Please check if the download completed successfully.")

### 1. Pixel-Level Contrastive Loss
This loss function takes the dense feature maps from the two augmented views and pulls features from the same spatial location (positive pairs) together, while pushing them away from features in other locations or classes (negative pairs).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PixelContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super(PixelContrastiveLoss, self).__init__()
        self.temperature = temperature
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, z_i, z_j, mask):
        # z_i, z_j shape: (Batch, Channels, Height, Width)
        B, C, H, W = z_i.shape

        # Flatten spatial dimensions -> (Batch, Channels, H*W)
        z_i_flat = z_i.view(B, C, -1)
        z_j_flat = z_j.view(B, C, -1)

        # Rearrange to (Batch * H * W, Channels) to treat each pixel as a sample
        z_i_flat = z_i_flat.permute(0, 2, 1).contiguous().view(-1, C)
        z_j_flat = z_j_flat.permute(0, 2, 1).contiguous().view(-1, C)

        # For simplicity in this skeleton, we treat corresponding spatial pixels as positive pairs.
        # In a full implementation, you would use `mask` to sample specific classes (like boundaries)
        # and only compute loss on those sampled pixels to save memory.

        # Number of total pixels
        N = z_i_flat.shape[0]

        # Similarity matrix (N x N)
        # Note: In practice, N is too large for full matrix multiplication (e.g., 256x256 = 65536 pixels).
        # You must sample a subset of pixels (e.g., 1024 random pixels or boundary pixels) per batch.
        # Here we simulate with a small subset to prevent OOM errors.

        sample_size = min(N, 1024)
        indices = torch.randperm(N)[:sample_size]

        z_i_sub = z_i_flat[indices]
        z_j_sub = z_j_flat[indices]

        # Calculate InfoNCE on the subset
        z_both = torch.cat([z_i_sub, z_j_sub], dim=0)
        sim_matrix = torch.matmul(z_both, z_both.T) / self.temperature

        labels = torch.cat([torch.arange(sample_size) + sample_size, torch.arange(sample_size)], dim=0).to(z_i.device)

        # Mask out self-similarity
        mask_self = torch.eye(2 * sample_size, dtype=torch.bool).to(z_i.device)
        sim_matrix.masked_fill_(mask_self, -9e15)

        loss = self.criterion(sim_matrix, labels)
        return loss

pixel_loss_fn = PixelContrastiveLoss()
print("Pixel-Level Contrastive Loss initialized!")

### 2. INbreast Dataset Preprocessing & Augmentations
For contrastive learning, your dataset needs to return *two* augmented versions of the same image. We use `torchvision.transforms` (or `albumentations` for more advanced spatial+mask augmentations) to apply random crops, flips, and color jitters.

### 6. TransUNet Decoder
The decoder's job is to take the dense, context-rich feature map from the ViT bottleneck (e.g., `32x32`) and progressively upsample it back to the original image resolution (e.g., `256x256`), outputting the final class probabilities for each pixel.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TransUNetDecoder(nn.Module):
    def __init__(self, in_channels=512, num_classes=2):
        super(TransUNetDecoder, self).__init__()

        # Upsampling block 1: 32x32 -> 64x64
        self.up1 = nn.ConvTranspose2d(in_channels, 256, kernel_size=2, stride=2)
        self.conv1 = nn.Sequential(
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )

        # Upsampling block 2: 64x64 -> 128x128
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = nn.Sequential(
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )

        # Upsampling block 3: 128x128 -> 256x256
        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        # Final output layer
        self.final_conv = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        # x shape from encoder: (B, 512, 32, 32)
        x = self.up1(x)
        x = self.conv1(x)

        x = self.up2(x)
        x = self.conv2(x)

        x = self.up3(x)
        x = self.conv3(x)

        out = self.final_conv(x)
        return out

# Test the Decoder
decoder = TransUNetDecoder(in_channels=512, num_classes=2)
dummy_encoded = torch.randn(2, 512, 32, 32)
decoder_out = decoder(dummy_encoded)

print(f"Encoder output shape (input to decoder): {dummy_encoded.shape}")
print(f"Final Segmentation Map shape: {decoder_out.shape}")
print("Decoder initialized successfully!")

### 7. Training Loop for Dense Contrastive Pre-training
This loop focuses *only* on the contrastive pre-training phase (Approach B). We use the `TransUNetEncoder` and the `DenseContrastiveModel` projector. The goal is to learn powerful, boundary-aware feature representations before fine-tuning the full segmentation model.

### 8. Save Processed Dataset for Later
Processing DICOM and XML files on the fly can be slow. Here, we iterate through your dataset, extract the raw images and masks, and save them directly as PyTorch tensors `.pt` files to your Google Drive.

In [ ]:
import os
import torch
from tqdm import tqdm

# Directory to save the processed tensors
save_dir = '/content/drive/MyDrive/INbreast_Processed'
os.makedirs(save_dir, exist_ok=True)

# Assuming dcm_files is available from earlier
# Note: Update xml_dir to the actual path where your Kaggle XMLs are extracted
xml_dir = '/content/inbreast/AllXML' # Placeholder, please verify the extracted path
os.makedirs(xml_dir, exist_ok=True) # Create dummy dir to avoid crash in this skeleton

print(f"Saving processed dataset to {save_dir}...")

# Process ALL files by passing the entire dcm_files list
dataset = INbreastDenseContrastiveDataset(dcm_files=dcm_files, xml_dir=xml_dir, image_size=256)

for i in tqdm(range(len(dataset))):
    # For fine-tuning, we only strictly need the image and the mask.
    # The dataset returns augmented views, but for the saved base dataset,
    # you would ideally save the unaugmented base tensors.
    v1_img, v1_mask, _, _ = dataset[i]

    # Save as dictionary
    data_dict = {
        'image': v1_img,
        'mask': v1_mask
    }
    torch.save(data_dict, os.path.join(save_dir, f'sample_{i}.pt'))

print("Dataset successfully processed and saved!")

### 9. Full TransUNet Model & Fine-Tuning Loop
Now we combine the pre-trained `TransUNetEncoder` and the `TransUNetDecoder` into a single model. We then train this full model using a standard segmentation loss (like BCEWithLogitsLoss) on the ground truth masks.

### 11. Evaluation Metrics (Dice Score & IoU)
These are the standard metrics for medical image segmentation. We compute them by comparing the model's predicted binary mask to the ground truth mask.

### 12. Full Training Loop with Model Checkpointing
Now we put it all together. This function trains the model, evaluates it on the validation set, and saves the weights of the best performing model based on the Dice score.

In [ ]:
import torch
import pandas as pd
from IPython.display import display

def calculate_confusion_matrix_elements(pred_mask, true_mask):
    # Flatten the masks to compute pixel-wise metrics
    pred_flat = pred_mask.contiguous().view(-1).long()
    true_flat = true_mask.contiguous().view(-1).long()

    # Calculate True Positives, False Positives, True Negatives, False Negatives
    tp = torch.sum((pred_flat == 1) & (true_flat == 1)).item()
    fp = torch.sum((pred_flat == 1) & (true_flat == 0)).item()
    tn = torch.sum((pred_flat == 0) & (true_flat == 0)).item()
    fn = torch.sum((pred_flat == 0) & (true_flat == 1)).item()

    return tp, fp, tn, fn

def generate_evaluation_matrix(model, dataloader, device):
    model.eval()
    total_tp, total_fp, total_tn, total_fn = 0, 0, 0, 0

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device)

            # Forward pass
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            # Accumulate confusion matrix elements
            tp, fp, tn, fn = calculate_confusion_matrix_elements(preds, masks)
            total_tp += tp
            total_fp += fp
            total_tn += tn
            total_fn += fn

    # Calculate specific metrics (adding epsilon to avoid division by zero)
    epsilon = 1e-6
    accuracy = (total_tp + total_tn) / (total_tp + total_fp + total_tn + total_fn + epsilon)
    precision = total_tp / (total_tp + total_fp + epsilon)
    recall = total_tp / (total_tp + total_fn + epsilon) # Also known as Sensitivity
    specificity = total_tn / (total_tn + total_fp + epsilon)
    dice_f1 = 2 * total_tp / (2 * total_tp + total_fp + total_fn + epsilon)
    iou = total_tp / (total_tp + total_fp + total_fn + epsilon)

    # Print raw pixel counts
    print("=== Pixel-Level Confusion Matrix ===")
    print(f"True Positives (TP) : {total_tp}")
    print(f"False Positives (FP): {total_fp}")
    print(f"True Negatives (TN) : {total_tn}")
    print(f"False Negatives (FN): {total_fn}\n")

    # Create a DataFrame to act as our Evaluation Matrix
    metrics_dict = {
        "Evaluation Metric": ["Pixel Accuracy", "Precision", "Recall (Sensitivity)", "Specificity", "Dice Score (F1)", "IoU (Jaccard Index)"],
        "Score": [f"{accuracy:.4f}", f"{precision:.4f}", f"{recall:.4f}", f"{specificity:.4f}", f"{dice_f1:.4f}", f"{iou:.4f}"]
    }

    df_metrics = pd.DataFrame(metrics_dict)
    print("=== Segmentation Performance Matrix ===")
    display(df_metrics)

    return df_metrics

# Run the evaluation matrix on the validation set using our trained model
print("Evaluating full_model on validation dataloader...")
eval_matrix_df = generate_evaluation_matrix(full_model, val_loader, device)


In [ ]:
import torch

# 1. Load the best saved model weights
checkpoint_path = '/content/drive/MyDrive/INbreast_Project/Checkpoints/best_transunet_model.pth'
full_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
print("Successfully loaded best model weights from Drive!")

# 2. Get a batch from the validation dataloader
val_images, val_masks = next(iter(val_loader))

# 3. Select the first image and mask from the batch
sample_image = val_images[0:1] # keep batch dimension: (1, 1, 256, 256)
sample_mask = val_masks[0:1]   # keep batch dimension: (1, 256, 256)

# 4. Visualize the prediction
print("\nVisualizing Prediction on Real Validation Sample:")
visualize_prediction(full_model, sample_image, sample_mask, device)


### 14. Run the Full Fine-Tuning Process
Let's start the actual fine-tuning process on the loaded dataset. The best model weights will be saved to your Drive based on the Validation Dice score.

In [ ]:
import matplotlib.pyplot as plt

# Define the number of epochs to train
num_epochs = 50

print(f"Starting Full Fine-Tuning for {num_epochs} epochs...")

# Run the training and validation loop
best_val_dice, history = train_and_validate(
    model=full_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    epochs=num_epochs,
    device=device
)

print(f"\nProcess finished! Best Validation Dice Score achieved: {best_val_dice:.4f}")

# Plotting the metrics
epochs_range = range(1, num_epochs + 1)
plt.figure(figsize=(10, 5))
plt.plot(epochs_range, history['val_dice'], label='Validation Dice Score', marker='o', color='blue')
plt.plot(epochs_range, history['val_iou'], label='Validation IoU', marker='s', color='orange')
plt.title('Validation Metrics Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Score')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import torch
import pandas as pd
from IPython.display import display
from torch.utils.data import DataLoader, random_split

def calculate_confusion_matrix_elements(pred_mask, true_mask):
    pred_flat = pred_mask.contiguous().view(-1).long()
    true_flat = true_mask.contiguous().view(-1).long()

    tp = torch.sum((pred_flat == 1) & (true_flat == 1)).item()
    fp = torch.sum((pred_flat == 1) & (true_flat == 0)).item()
    tn = torch.sum((pred_flat == 0) & (true_flat == 0)).item()
    fn = torch.sum((pred_flat == 0) & (true_flat == 1)).item()

    return tp, fp, tn, fn

def generate_evaluation_matrix(model, dataloader, device):
    model.eval()
    total_tp, total_fp, total_tn, total_fn = 0, 0, 0, 0

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            tp, fp, tn, fn = calculate_confusion_matrix_elements(preds, masks)
            total_tp += tp
            total_fp += fp
            total_tn += tn
            total_fn += fn

    epsilon = 1e-6
    accuracy = (total_tp + total_tn) / (total_tp + total_fp + total_tn + total_fn + epsilon)
    precision = total_tp / (total_tp + total_fp + epsilon)
    recall = total_tp / (total_tp + total_fn + epsilon)
    specificity = total_tn / (total_tn + total_fp + epsilon)
    dice_f1 = 2 * total_tp / (2 * total_tp + total_fp + total_fn + epsilon)
    iou = total_tp / (total_tp + total_fp + total_fn + epsilon)

    print("=== Pixel-Level Confusion Matrix ===")
    print(f"True Positives (TP) : {total_tp}")
    print(f"False Positives (FP): {total_fp}")
    print(f"True Negatives (TN) : {total_tn}")
    print(f"False Negatives (FN): {total_fn}\n")

    metrics_dict = {
        "Evaluation Metric": ["Pixel Accuracy", "Precision", "Recall (Sensitivity)", "Specificity", "Dice Score (F1)", "IoU (Jaccard Index)"],
        "Score": [f"{accuracy:.4f}", f"{precision:.4f}", f"{recall:.4f}", f"{specificity:.4f}", f"{dice_f1:.4f}", f"{iou:.4f}"]
    }

    df_metrics = pd.DataFrame(metrics_dict)
    print("=== Segmentation Performance Matrix ===")
    display(df_metrics)
    return df_metrics

# 1. Re-initialize the Dataset and DataLoaders on the full processed data
processed_dir = '/content/drive/MyDrive/INbreast_Processed'
full_dataset = ProcessedINbreastDataset(processed_dir)

print(f"Total processed samples found: {len(full_dataset)}")

if len(full_dataset) > 10:
    # Split into Train (80%) and Validation (20%)
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size

    generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

    batch_size = 4
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    print(f"Training set: {len(train_dataset)} samples")
    print(f"Validation set: {len(val_dataset)} samples")

    # 2. Evaluate the model on the full validation set
    print("\nEvaluating model on the full validation dataset...")
    eval_matrix_df = generate_evaluation_matrix(full_model, val_loader, device)
else:
    print("It looks like the full dataset hasn't finished processing yet. Please run the processing cell first!")


### 13. Create Train and Validation DataLoaders
Now we load the preprocessed dataset from Google Drive, split it into training and validation sets, and create `DataLoader` instances to feed the data into our model in batches.

In [ ]:
import glob
import torch
import os
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

class ProcessedINbreastDataset(Dataset):
    def __init__(self, data_dir):
        self.file_paths = glob.glob(os.path.join(data_dir, '*.pt'))

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # Load the saved dictionary
        data = torch.load(self.file_paths[idx])

        # The image and mask are (1, H, W)
        img = data['image']
        mask = data['mask']

        # Force resize to (256, 256) to fix dimension mismatches during batching
        img = F.interpolate(img.unsqueeze(0), size=(256, 256), mode='bilinear', align_corners=False).squeeze(0)

        # Our loss function and evaluation expect the mask to be (H, W) integer class indices (0 or 1).
        mask = F.interpolate(mask.unsqueeze(0), size=(256, 256), mode='nearest').squeeze(0).squeeze(0).long()

        return img, mask

# 1. Initialize the dataset using the directory we saved the tensors to
processed_dir = '/content/drive/MyDrive/INbreast_Processed'
full_dataset = ProcessedINbreastDataset(processed_dir)

# Ensure we have data
if len(full_dataset) == 0:
    print(f"Warning: No processed files found in {processed_dir}. Please check step 8.")
else:
    print(f"Total processed samples found: {len(full_dataset)}")

    # 2. Split into Train (80%) and Validation (20%)
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size

    # Use a fixed generator seed for reproducible splits
    generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)

    # 3. Create DataLoaders
    batch_size = 4
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    print(f"Training set: {len(train_dataset)} samples ({len(train_loader)} batches)")
    print(f"Validation set: {len(val_dataset)} samples ({len(val_loader)} batches)")
    print("DataLoaders are ready for training!")

In [ ]:
import os

# Directory to save model checkpoints in Google Drive
checkpoint_dir = '/content/drive/MyDrive/INbreast_Project/Checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

def train_and_validate(model, train_loader, val_loader, optimizer, criterion, epochs, device):
    best_dice = 0.0
    history = {'train_loss': [], 'val_dice': [], 'val_iou': []}

    for epoch in range(epochs):
        print(f"\n--- Epoch {epoch+1}/{epochs} ---")

        # --- Training Phase ---
        model.train()
        train_loss = 0.0
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)

            optimizer.zero_grad()
            outputs = model(images) # outputs shape: (B, 2, 256, 256)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader) if len(train_loader) > 0 else 0
        print(f"Training Loss: {avg_train_loss:.4f}")
        history['train_loss'].append(avg_train_loss)

        # --- Validation Phase ---
        print("Running Validation...")
        avg_dice, avg_iou = evaluate_model(model, val_loader, device)
        history['val_dice'].append(avg_dice)
        history['val_iou'].append(avg_iou)

        # --- Model Checkpointing ---
        if avg_dice > best_dice:
            print(f"Validation Dice improved from {best_dice:.4f} to {avg_dice:.4f}. Saving model...")
            best_dice = avg_dice
            checkpoint_path = os.path.join(checkpoint_dir, 'best_transunet_model.pth')
            torch.save(model.state_dict(), checkpoint_path)

    print("\nTraining complete. Best model saved to Drive!")
    return best_dice, history

print("Full training loop with checkpointing is ready!")

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

def calculate_metrics(pred_mask, true_mask):
    """
    Calculates the Dice Coefficient and Intersection over Union (IoU).
    pred_mask: Predicted binary mask (B, H, W)
    true_mask: Ground truth binary mask (B, H, W)
    """
    # Flatten the tensors
    pred_flat = pred_mask.contiguous().view(-1).float()
    true_flat = true_mask.contiguous().view(-1).float()

    # Calculate Intersection and Union
    intersection = torch.sum(pred_flat * true_flat)

    # Add a small epsilon (1e-6) to avoid division by zero
    dice = (2. * intersection + 1e-6) / (torch.sum(pred_flat) + torch.sum(true_flat) + 1e-6)

    union = torch.sum(pred_flat) + torch.sum(true_flat) - intersection
    iou = (intersection + 1e-6) / (union + 1e-6)

    return dice.item(), iou.item()

def evaluate_model(model, val_loader, device):
    model.eval() # Set model to evaluation mode (important for BatchNorm/Dropout)
    total_dice = 0.0
    total_iou = 0.0
    num_batches = 0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            # Forward pass
            outputs = model(images) # Shape: (B, 2, 256, 256)

            # Get predictions (argmax over class dimension)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1) # Shape: (B, 256, 256)

            # Calculate metrics for this batch
            dice, iou = calculate_metrics(preds, masks)
            total_dice += dice
            total_iou += iou
            num_batches += 1

    avg_dice = total_dice / num_batches if num_batches > 0 else 0
    avg_iou = total_iou / num_batches if num_batches > 0 else 0

    print(f"Validation Results -> Average Dice Score: {avg_dice:.4f} | Average IoU: {avg_iou:.4f}")
    return avg_dice, avg_iou

# --- Test Evaluation with Dummy Data ---
print("Testing Validation Evaluation...")
# Create a dummy validation dataset and loader
dummy_val_images = torch.randn(8, 1, 256, 256)
dummy_val_masks = torch.randint(0, 2, (8, 256, 256)).long()

dummy_val_dataset = TensorDataset(dummy_val_images, dummy_val_masks)
dummy_val_loader = DataLoader(dummy_val_dataset, batch_size=4, shuffle=False)

# Run evaluation
avg_val_dice, avg_val_iou = evaluate_model(full_model, dummy_val_loader, device)

### 10. Inference and Visualization
After fine-tuning the model, we need to evaluate it on unseen data. The function below runs a forward pass on a validation image and visualizes the Original Image, the Ground Truth Mask, and the Predicted Mask side-by-side.

In [ ]:
import matplotlib.pyplot as plt
import torch

def visualize_prediction(model, image_tensor, mask_tensor, device):
    """
    Runs inference on a single image and plots the results.
    image_tensor: shape (1, 1, 256, 256)
    mask_tensor: shape (1, 256, 256)
    """
    model.eval() # Set model to evaluation mode

    with torch.no_grad():
        # Move to device and get raw logits
        img = image_tensor.to(device)
        output = model(img) # Shape: (1, 2, 256, 256)

        # Convert logits to probabilities using Softmax, then get the predicted class (0 or 1)
        probs = torch.softmax(output, dim=1)
        pred_mask = torch.argmax(probs, dim=1).squeeze().cpu().numpy()

    # Convert inputs back to numpy for plotting
    img_np = image_tensor.squeeze().numpy()
    true_mask_np = mask_tensor.squeeze().numpy()

    # Plotting
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(img_np, cmap='gray')
    axes[0].set_title('Original Mammogram (Input)')
    axes[0].axis('off')

    axes[1].imshow(true_mask_np, cmap='gray')
    axes[1].set_title('Ground Truth Mask')
    axes[1].axis('off')

    axes[2].imshow(pred_mask, cmap='gray')
    axes[2].set_title('Model Prediction')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

# --- Test the Visualization with Dummy Data ---
print("Testing Inference Visualization...")
dummy_val_image = torch.randn(1, 1, 256, 256)
dummy_val_mask = torch.randint(0, 2, (1, 256, 256))

# Pass the dummy data through our recently trained full_model
visualize_prediction(full_model, dummy_val_image, dummy_val_mask, device)

In [ ]:
import torch.nn as nn
import torch.optim as optim

# --- 1. Combine Encoder and Decoder ---
class TransUNet(nn.Module):
    def __init__(self, encoder, decoder):
        super(TransUNet, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, x):
        features = self.encoder(x)
        out = self.decoder(features)
        return out

# Initialize the full model
# (In practice, you would load the pre-trained weights into the encoder first)
full_model = TransUNet(encoder, decoder).to(device)

# --- 2. Fine-Tuning Setup ---
criterion = nn.CrossEntropyLoss() # Assuming num_classes=2 (Background vs Tumor)
optimizer = optim.Adam(full_model.parameters(), lr=1e-4)

# --- 3. Fine-Tuning Loop Skeleton ---
num_finetune_epochs = 3
print("Starting Fine-Tuning phase...")

for epoch in range(num_finetune_epochs):
    full_model.train()
    epoch_loss = 0.0

    # Simulate a single batch of fine-tuning data
    # Shape: (Batch, Channels, H, W)
    images = torch.randn(4, 1, 256, 256).to(device)
    # Target masks for CrossEntropy should be (Batch, H, W) with class indices (0 or 1)
    masks = torch.randint(0, 2, (4, 256, 256)).long().to(device)

    optimizer.zero_grad()

    # Forward Pass
    outputs = full_model(images) # Shape: (B, 2, 256, 256)

    # Compute Segmentation Loss
    loss = criterion(outputs, masks)

    # Backward Pass & Optimize
    loss.backward()
    optimizer.step()

    epoch_loss += loss.item()
    print(f"Fine-Tune Epoch [{epoch+1}/{num_finetune_epochs}], Segmentation Loss: {epoch_loss:.4f}")

print("Fine-Tuning completed!")

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import numpy as np
import cv2

# 1. Initialize the dataset
# Using the previously defined INbreastDenseContrastiveDataset
dataset = INbreastDenseContrastiveDataset(dcm_files=dcm_files, xml_dir=xml_dir, image_size=256)

# 2. Find a sample that actually contains a tumor (mask sum > 0)
sample_found = False
for i in range(min(len(dataset), 50)): # Limit search to save time
    try:
        v1_img, v1_mask, v2_img, v2_mask = dataset[i]
        if v1_mask.sum() > 0: # Ensure there is a tumor in the view
            sample_found = True
            break
    except Exception as e:
        # Skip DICOM files that are missing pixel data or corrupted
        continue

# Fallback to synthetic data if no real tumor is found in the current subset
if not sample_found:
    print("Could not find a valid tumor mask in the subset. Generating a synthetic example...")
    # Synthetic View 1
    v1_img = torch.rand(1, 256, 256)
    v1_mask_np = np.zeros((256, 256), dtype=np.float32)
    cv2.circle(v1_mask_np, (128, 128), 40, 1, -1)
    v1_mask = torch.tensor(v1_mask_np).unsqueeze(0)

    # Synthetic View 2 (slightly transformed)
    v2_img = torch.rand(1, 256, 256)
    v2_mask_np = np.zeros((256, 256), dtype=np.float32)
    cv2.circle(v2_mask_np, (140, 110), 45, 1, -1) # Shifted and scaled
    v2_mask = torch.tensor(v2_mask_np).unsqueeze(0)

# 3. Function to extract boundary pixels using PyTorch operations
def get_boundary(mask_tensor):
    # Add batch and channel dimensions: (1, 1, H, W)
    mask_float = mask_tensor.unsqueeze(0).float()

    # Dilation (expands the mask)
    dilated = F.max_pool2d(mask_float, kernel_size=5, stride=1, padding=2)
    # Erosion (shrinks the mask)
    eroded = -F.max_pool2d(-mask_float, kernel_size=5, stride=1, padding=2)

    # Boundary is the difference between dilated and eroded masks
    boundary = dilated - eroded
    return boundary.squeeze().numpy()

# Get boundaries for both views
v1_boundary = get_boundary(v1_mask)
v2_boundary = get_boundary(v2_mask)

# 4. Visualize the results
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# View 1 Plotting
axes[0, 0].imshow(v1_img.squeeze().numpy(), cmap='gray')
axes[0, 0].set_title('View 1: Augmented Image')
axes[0, 0].axis('off')

axes[0, 1].imshow(v1_mask.squeeze().numpy(), cmap='gray')
axes[0, 1].set_title('View 1: Full Tumor Mask')
axes[0, 1].axis('off')

axes[0, 2].imshow(v1_boundary, cmap='hot')
axes[0, 2].set_title('View 1: Tumor Boundary Pixels')
axes[0, 2].axis('off')

# View 2 Plotting
axes[1, 0].imshow(v2_img.squeeze().numpy(), cmap='gray')
axes[1, 0].set_title('View 2: Augmented Image')
axes[1, 0].axis('off')

axes[1, 1].imshow(v2_mask.squeeze().numpy(), cmap='gray')
axes[1, 1].set_title('View 2: Full Tumor Mask')
axes[1, 1].axis('off')

axes[1, 2].imshow(v2_boundary, cmap='hot')
axes[1, 2].set_title('View 2: Tumor Boundary Pixels')
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
import torch

# 1. Load the best saved model weights from the 50-epoch run
checkpoint_path = '/content/drive/MyDrive/INbreast_Project/Checkpoints/best_transunet_model.pth'
full_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
print("Successfully loaded best model weights from Drive!")

# 2. Run the Quantitative Evaluation (Confusion Matrix & Metrics)
print("\n--- Quantitative Analysis (Validation Set) ---")
# Make sure val_loader is defined from the previous cells
eval_matrix_df = generate_evaluation_matrix(full_model, val_loader, device)

# 3. Run the Qualitative Visualization
print("\n--- Qualitative Analysis (Visualizing Predictions) ---")
# Get a batch from the validation dataloader
val_images, val_masks = next(iter(val_loader))

# Select the first image and mask from the batch
sample_image = val_images[0:1] # keep batch dimension: (1, 1, 256, 256)
sample_mask = val_masks[0:1]   # keep batch dimension: (1, 256, 256)

# Visualize the prediction
visualize_prediction(full_model, sample_image, sample_mask, device)


In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader

# --- 1. Initialization ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize Encoder and the Dense Projector Model
encoder = TransUNetEncoder(in_channels=1, hidden_dim=512).to(device)
dense_model = DenseContrastiveModel(encoder, in_channels=512, out_dim=128).to(device)

# Initialize Loss and Optimizer
criterion = PixelContrastiveLoss(temperature=0.5).to(device)
optimizer = optim.Adam(dense_model.parameters(), lr=1e-4)

# Initialize a dummy dataloader for demonstration (replace with real INbreast dataset later)
# dataset = INbreastDenseContrastiveDataset(dcm_files=dcm_files, xml_dir=xml_dir)
# dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# --- 2. Dummy Training Loop Skeleton ---
num_epochs = 3
print("Starting Dense Contrastive Pre-training...")

for epoch in range(num_epochs):
    dense_model.train()
    epoch_loss = 0.0

    # Simulate a single batch of data: (View1_img, View1_mask, View2_img, View2_mask)
    # Shape: (Batch, Channels, H, W)
    v1_img = torch.randn(4, 1, 256, 256).to(device)
    v1_mask = torch.randint(0, 2, (4, 1, 256, 256)).float().to(device)
    v2_img = torch.randn(4, 1, 256, 256).to(device)
    v2_mask = torch.randint(0, 2, (4, 1, 256, 256)).float().to(device)

    optimizer.zero_grad()

    # Forward Pass through DenseContrastiveModel
    z1 = dense_model(v1_img) # (B, 128, 32, 32)
    z2 = dense_model(v2_img) # (B, 128, 32, 32)

    # Compute Pixel-Level Contrastive Loss
    # Note: In a full implementation, you'd pass the masks to strategically sample positive/negative pixels
    loss = criterion(z1, z2, mask=v1_mask)

    # Backward Pass & Optimize
    loss.backward()
    optimizer.step()

    epoch_loss += loss.item()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

print("Pre-training completed!")

In [ ]:
import matplotlib.pyplot as plt
import torch

# Ensure the Dense Contrastive Model is in evaluation mode
dense_model.eval()

print("Extracting a sample from the training set...")
# Get a batch from the training loader (since our val set lacked tumors)
train_images, train_masks = next(iter(train_loader))
sample_img = train_images[0:1].to(device) # Keep batch dimension: (1, 1, 256, 256)

with torch.no_grad():
    # Pass the image through the encoder and 1x1 projector to get the dense feature map
    # Output shape will be (Batch, Projection_Dim, H_feature, W_feature)
    dense_feature_map = dense_model(sample_img)

print(f"Input Image Shape: {sample_img.shape}")
print(f"Dense Feature Map Shape: {dense_feature_map.shape}")
print("Notice how spatial dimensions (32x32) are preserved, but we now have 128 channels of features per 'pixel'!")

# Visualize the original image and the first 3 channels of the dense feature map
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Plot Original Image
axes[0].imshow(sample_img.cpu().squeeze().numpy(), cmap='gray')
axes[0].set_title('Original Input Image (256x256)')
axes[0].axis('off')

# Plot the first 3 channels of the dense feature map
for i in range(3):
    # Extract a single channel from the dense feature map
    feature_channel = dense_feature_map[0, i, :, :].cpu().numpy()
    axes[i+1].imshow(feature_channel, cmap='viridis')
    axes[i+1].set_title(f'Dense Feature Map\nChannel {i+1} (32x32)')
    axes[i+1].axis('off')

plt.tight_layout()
plt.show()

### Understanding Dice Score and IoU

To calculate these metrics manually, we first need to break down the pixel predictions into three categories (ignoring True Negatives for these specific metrics):

*   **True Positives (TP):** The model predicted 'Tumor' (1), and the ground truth is 'Tumor' (1).
*   **False Positives (FP):** The model predicted 'Tumor' (1), but the ground truth is 'Background' (0).
*   **False Negatives (FN):** The model predicted 'Background' (0), but the ground truth is 'Tumor' (1).

#### 1. Intersection over Union (IoU) / Jaccard Index
IoU measures the exact overlap between the predicted mask and the true mask, divided by the total area covered by *both* masks combined.
*   **Formula:** `IoU = TP / (TP + FP + FN)`

#### 2. Dice Score / F1 Score
The Dice Score is very similar to IoU but gives exactly twice the weight to the True Positives. It is the most common metric in medical imaging.
*   **Formula:** `Dice = (2 * TP) / (2 * TP + FP + FN)`

*Note: We usually add a tiny number (like `1e-6`, called epsilon) to both the numerator and denominator to prevent a "Division by Zero" error when an image has no tumors.*

In [ ]:
import torch

def calculate_manual_metrics(pred_mask, true_mask, epsilon=1e-6):
    """
    Manually calculates Dice Score and IoU using explicit TP, FP, and FN counts.
    Assumes inputs are binary masks (0 or 1) of the same shape.
    """
    # Flatten the tensors to 1D arrays for easy comparison
    pred_flat = pred_mask.contiguous().view(-1).float()
    true_flat = true_mask.contiguous().view(-1).float()

    # Calculate True Positives (Intersection)
    tp = torch.sum(pred_flat * true_flat)

    # Calculate False Positives (Predicted 1, True 0)
    fp = torch.sum(pred_flat * (1 - true_flat))

    # Calculate False Negatives (Predicted 0, True 1)
    fn = torch.sum((1 - pred_flat) * true_flat)

    # Calculate Metrics
    dice = (2 * tp + epsilon) / (2 * tp + fp + fn + epsilon)
    iou = (tp + epsilon) / (tp + fp + fn + epsilon)

    print(f"True Positives (TP): {tp.item()}")
    print(f"False Positives (FP): {fp.item()}")
    print(f"False Negatives (FN): {fn.item()}")
    print(f"---> Calculated Dice Score: {dice.item():.4f}")
    print(f"---> Calculated IoU Score: {iou.item():.4f}\n")

    return dice.item(), iou.item()

# --- Let's test it with a synthetic example ---
print("Scenario 1: Partial Overlap")
dummy_true = torch.tensor([1, 1, 1, 0, 0])
dummy_pred = torch.tensor([1, 1, 0, 1, 0])
calculate_manual_metrics(dummy_pred, dummy_true)

print("Scenario 2: Empty Image (Like our previous validation batch)")
empty_true = torch.tensor([0, 0, 0, 0, 0])
empty_pred = torch.tensor([0, 0, 0, 0, 0])
calculate_manual_metrics(empty_pred, empty_true)

In [ ]:
import torch.nn as nn
import torch.optim as optim

# 1. Ensure the whole model (Encoder + Decoder) is unfrozen
for param in full_model.parameters():
    param.requires_grad = True

print("All layers in the Encoder and Decoder are unfrozen and ready for fine-tuning.")

# 2. Define the Standard Segmentation Loss (Cross Entropy)
# Since our model outputs shape (Batch, 2, H, W) and our masks are (Batch, H, W) with indices 0 or 1
segmentation_criterion = nn.CrossEntropyLoss()

# 3. Define the Optimizer (Adam is standard)
# We pass full_model.parameters() so both encoder and decoder get updated
optimizer = optim.Adam(full_model.parameters(), lr=1e-4)

# 4. Run the Fine-Tuning Loop
fine_tune_epochs = 5 # Set to a small number for demonstration
print(f"Starting Full Model Fine-Tuning for {fine_tune_epochs} epochs...")

# Using the train_and_validate function we defined earlier
best_val_dice, history = train_and_validate(
    model=full_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=segmentation_criterion,
    epochs=fine_tune_epochs,
    device=device
)

print(f"\nFine-tuning completed! Best Validation Dice: {best_val_dice:.4f}")

In [ ]:
import torch

print("Searching for a validation sample with a tumor mask...")
found_tumor = False

# Iterate through the validation dataset
for i in range(len(val_dataset)):
    img, mask = val_dataset[i]

    # Check if the mask has any positive pixels (tumor)
    if mask.sum() > 0:
        print(f"Tumor found in validation sample index: {i}")

        # Add batch dimension as expected by the model and visualization function
        img_batch = img.unsqueeze(0) # Shape: (1, 1, 256, 256)
        mask_batch = mask.unsqueeze(0) # Shape: (1, 256, 256)

        # Visualize the prediction
        print("\n--- Qualitative Analysis (Positive Tumor Sample) ---")
        visualize_prediction(full_model, img_batch, mask_batch, device)

        found_tumor = True
        break # Stop after finding the first one

if not found_tumor:
    print("Could not find any images with a tumor in the current validation split.")
    print("This is common in small dataset splits or highly imbalanced datasets like INbreast where many cases are benign/normal.")

### 5. Integrating Masks into the Contrastive Dataset
For pixel-level contrastive learning, we need the ground truth masks to sample positive and negative pixel pairs. We use `torchvision.transforms.v2` to easily apply the identical spatial transformations to both the image and the mask.

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
import pydicom
import numpy as np
from PIL import Image

class INbreastDenseContrastiveDataset(Dataset):
    def __init__(self, dcm_files, xml_dir, image_size=256):
        self.dcm_files = dcm_files
        self.xml_dir = xml_dir
        self.image_size = image_size

        # Spatial transforms must apply identically to image AND mask
        self.spatial_transform = v2.Compose([
            v2.Resize((image_size, image_size), interpolation=v2.InterpolationMode.NEAREST),
            v2.RandomHorizontalFlip(p=0.5),
            v2.RandomVerticalFlip(p=0.5),
            v2.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1))
        ])

        # Color transforms only apply to the image, not the mask
        self.color_transform = v2.Compose([
            v2.ColorJitter(brightness=0.3, contrast=0.3),
            v2.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))
        ])

    def __len__(self):
        return len(self.dcm_files)

    def __getitem__(self, idx):
        # 1. Load DICOM Image
        dcm_path = self.dcm_files[idx]
        dcm_filename = os.path.basename(dcm_path)

        # Use force=True to bypass missing DICOM headers
        dicom_data = pydicom.dcmread(dcm_path, force=True)

        # If forcing read, we might need to set the transfer syntax manually to extract pixel_array
        if 'TransferSyntaxUID' not in dicom_data.file_meta:
            dicom_data.file_meta.TransferSyntaxUID = pydicom.uid.ImplicitVRLittleEndian

        image = dicom_data.pixel_array

        # Normalize image to 0-1
        image = (image - np.min(image)) / (np.max(image) - np.min(image) + 1e-8)
        image_tensor = torch.tensor(image, dtype=torch.float32).unsqueeze(0) # (1, H, W)

        # 2. Load Mask (Assuming XML filenames match DCM filenames up to the extension/ID)
        image_id = dcm_filename.split('_')[0]
        xml_path = os.path.join(self.xml_dir, f"{image_id}.xml")

        # Use the mask parsing function defined earlier
        mask_np = load_inbreast_mask(xml_path, image.shape)
        mask_tensor = torch.tensor(mask_np, dtype=torch.float32).unsqueeze(0) # (1, H, W)

        # 3. Generate View 1 (Image + Mask)
        view1_img, view1_mask = self.spatial_transform(image_tensor, mask_tensor)
        view1_img = self.color_transform(view1_img)

        # 4. Generate View 2 (Image + Mask)
        view2_img, view2_mask = self.spatial_transform(image_tensor, mask_tensor)
        view2_img = self.color_transform(view2_img)

        return view1_img, view1_mask, view2_img, view2_mask

print("Dense Contrastive Dataset class with integrated masks is ready and fixed for missing headers!")

### 4. TransUNet Encoder Skeleton
This hybrid architecture combines a CNN to extract local features with a Vision Transformer (ViT) to capture global spatial context. The output is a dense feature map that feeds directly into our `DenseContrastiveModel`.

In [ ]:
import torch
import torch.nn as nn

class TransUNetEncoder(nn.Module):
    def __init__(self, in_channels=1, hidden_dim=512, num_layers=4, num_heads=8):
        super(TransUNetEncoder, self).__init__()

        # 1. CNN Backbone (Feature Extraction)
        # Downsamples the image and extracts local features
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2), # e.g., 256 -> 128

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2), # 128 -> 64

            nn.Conv2d(128, hidden_dim, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)  # 64 -> 32
        )

        # 2. Vision Transformer (Global Context)
        # Processes the CNN feature map as a sequence of patches
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim*4,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x):
        # Extract CNN features
        features = self.cnn(x) # Shape: (B, hidden_dim, H, W)
        B, C, H, W = features.shape

        # Flatten spatial dimensions to create sequence of patches (B, H*W, C)
        patches = features.view(B, C, -1).permute(0, 2, 1)

        # Apply Transformer
        transformed_patches = self.transformer(patches)

        # Reshape back to image format (B, C, H, W)
        out = transformed_patches.permute(0, 2, 1).view(B, C, H, W)

        return out

# Let's test it with a dummy image batch (2 images, 1 channel, 256x256)
dummy_input = torch.randn(2, 1, 256, 256)
encoder = TransUNetEncoder(in_channels=1)
output_features = encoder(dummy_input)

print(f"Input image shape: {dummy_input.shape}")
print(f"Encoder output features shape: {output_features.shape}")
print("TransUNet Encoder initialized and tested successfully!")

### 3. Parsing INbreast XML Annotations into Masks
INbreast annotations are stored as `plist` (Property List) XML files. The function below reads the polygon coordinates for 'Mass' lesions and converts them into a 2D numpy array mask.

In [ ]:
import plistlib
import numpy as np
import cv2
import os

def load_inbreast_mask(xml_path, image_shape):
    """
    Parses an INbreast XML annotation file and returns a binary mask.

    Args:
        xml_path (str): Path to the XML file.
        image_shape (tuple): The shape of the corresponding DICOM image (H, W).

    Returns:
        np.ndarray: A binary mask (1 for mass, 0 for background).
    """
    # Create an empty mask of the same size as the image
    mask = np.zeros(image_shape, dtype=np.uint8)

    if not os.path.exists(xml_path):
        # Return empty mask if no annotation exists for this image (e.g., normal cases)
        return mask

    with open(xml_path, 'rb') as f:
        plist_dict = plistlib.load(f)

    # Iterate through all ROIs in the XML
    for roi in plist_dict.get('Images', [{}])[0].get('ROIs', []):
        # We usually focus on 'Mass' for tumor segmentation
        if roi.get('Name') == 'Mass':
            points = roi.get('Point_px')
            if not points:
                continue

            # Extract (x, y) coordinates
            polygon = []
            for point in points:
                # Points are stored as strings: "(x, y)"
                x, y = eval(point)
                polygon.append([int(x), int(y)])

            # Fill the polygon on the mask
            polygon = np.array(polygon, dtype=np.int32)
            cv2.fillPoly(mask, [polygon], color=1)

    return mask

print("Mask parsing function defined! You can now integrate this into your Dataset class.")

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import pydicom
import numpy as np
from PIL import Image

class INbreastContrastiveDataset(Dataset):
    def __init__(self, dcm_files, mask_files=None, image_size=256):
        self.dcm_files = dcm_files
        self.mask_files = mask_files # In a real scenario, you'd load your XML/PNG masks here

        # Standard Augmentations for Contrastive Learning
        self.transform = T.Compose([
            T.ToPILImage(),
            T.Resize((image_size, image_size)),
            T.RandomHorizontalFlip(),
            T.RandomApply([T.ColorJitter(brightness=0.8, contrast=0.8)], p=0.8),
            T.RandomApply([T.GaussianBlur(kernel_size=9)], p=0.5),
            T.ToTensor(),
        ])

    def __len__(self):
        return len(self.dcm_files)

    def __getitem__(self, idx):
        # 1. Load DICOM
        dcm_path = self.dcm_files[idx]
        dicom_data = pydicom.dcmread(dcm_path)
        image = dicom_data.pixel_array

        # Normalize to 0-255 for PIL
        image = (image - np.min(image)) / (np.max(image) - np.min(image) + 1e-8)
        image = (image * 255).astype(np.uint8)

        # 2. Generate two different augmented views (X_i, X_j)
        view_1 = self.transform(image)
        view_2 = self.transform(image)

        # 3. Load Mask (Placeholder for now)
        # If you apply spatial transforms (like RandomCrop), you MUST apply the exact same
        # transform to the mask. Libraries like 'albumentations' are better for this.
        dummy_mask = torch.zeros((1, view_1.shape[1], view_1.shape[2]))

        return view_1, view_2, dummy_mask

# Let's test it with the files we found earlier
if 'dcm_files' in locals() and len(dcm_files) > 0:
    dataset = INbreastContrastiveDataset(dcm_files=dcm_files[:10]) # test with 10 files
    loader = DataLoader(dataset, batch_size=2, shuffle=True)

    v1, v2, m = next(iter(loader))
    print(f"View 1 shape: {v1.shape}")
    print(f"View 2 shape: {v2.shape}")
    print("Dataset initialized and tested successfully!")

In [ ]:
from google.colab import files

# The text content to save
concept_text = """### The Pixel-Level Contrastive Loss Concept
For the loss, you won't use the standard `NTXentLoss`. Instead, during training, you will:
1. Pass image `X` and its augmented version `X_aug` through the `DenseContrastiveModel`.
2. Use the **ground truth mask** to identify which pixels are "Tumor", "Boundary", or "Background".
3. Sample a few "Boundary" pixels from `X`. Their **positive pairs** are the exact same spatial pixels in `X_aug`.
4. Their **negative pairs** are pixels sampled from the "Background" or "Benign" regions within the same image or other images in the batch.
"""

# Save it as a text file in the Colab environment
file_name = "pixel_level_contrastive_loss_concept.txt"
with open(file_name, "w") as f:
    f.write(concept_text)

print(f"File '{file_name}' created successfully.")
print("Triggering download to your local machine...")

# Trigger the browser download
files.download(file_name)

### Approach B: Dense (Pixel-Level) Contrastive Learning Skeleton

Unlike image-level contrastive learning, we use a **1x1 Convolutional Projection Head**. This acts like an MLP applied independently to every single pixel's feature vector, preserving the spatial dimensions (Height and Width).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DenseContrastiveModel(nn.Module):
    def __init__(self, encoder, in_channels, out_dim=128):
        super(DenseContrastiveModel, self).__init__()
        self.encoder = encoder # This would be your U-Net or ViT encoder

        # Instead of nn.Linear, we use 1x1 Convs for dense pixel-wise projection
        self.projector = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(in_channels, out_dim, kernel_size=1)
        )

    def forward(self, x):
        # Extract spatial features (B, Channels, H, W)
        features = self.encoder(x)

        # Project to contrastive space (B, out_dim, H, W)
        z_dense = self.projector(features)

        # Normalize along the channel dimension for each pixel
        return F.normalize(z_dense, dim=1)

print("Dense Contrastive Model Skeleton defined!")

### The Pixel-Level Contrastive Loss Concept
For the loss, you won't use the standard `NTXentLoss`. Instead, during training, you will:
1. Pass image `X` and its augmented version `X_aug` through the `DenseContrastiveModel`.
2. Use the **ground truth mask** to identify which pixels are "Tumor", "Boundary", or "Background".
3. Sample a few "Boundary" pixels from `X`. Their **positive pairs** are the exact same spatial pixels in `X_aug`.
4. Their **negative pairs** are pixels sampled from the "Background" or "Benign" regions within the same image or other images in the batch.

## How to get the INbreast Dataset

### 1. The Official Academic Source
Originally, the dataset was released by the Breast Research Group at the University of Porto. To get the official, unmodified high-resolution DICOM files, you typically have to request access for research purposes by contacting the authors of the original 2012 paper.

### 2. The Easy Way: Kaggle (Recommended for Colab)
Many researchers have uploaded the INbreast dataset (often converted from DICOM to PNG/JPG for easier use in deep learning) to Kaggle.

**Direct Kaggle Link**: Search for [INbreast Dataset on Kaggle](https://www.kaggle.com/search?q=inbreast) (e.g., there are several versions containing the ROIs, masks, and full images).

### Fast-Track: Download directly to Colab via Kaggle API
Instead of downloading locally and uploading to Google Drive (which is slow), you can pull it directly into this Colab notebook.

**Steps:**
1. Go to your Kaggle account settings (kaggle.com -> Your Profile -> Account).
2. Click **"Create New API Token"**. This downloads a `kaggle.json` file to your computer.
3. Run the code cell below, and upload that `kaggle.json` file when prompted.